# <center>**Detector Spatial Resolution - Kernel Fit**<center>

- Let's fit the detector spatial resolution kernel parameters by applying the instr. effect to the source shadowgram model.

- Here the goal is to obtain the kernel parameters by convolving the source model with the PSFY kernel and minimising the MSE between convolved model and reconstructed detector images.

- The whole fit procedure is performed by dividing the energy range $[2, 30]$ keV in bands of 1 keV each to analyse the trend of the kernel parameters.<br>
The total range stops at 30 keV since above this value the kernel parameters variability is negligible.

- **NOTE**: The PSFY kernel is fitted by fixing the parameter $\beta$ (exponent inside the `modsech` argument), since from the tests in `demo_detector_spatial_res_KernelSingleParams.ipynb` better and more stable results have been obtained.<br>
This approach may be extended by including a small variability interval for $\beta$ around the value $\beta_{0} = 0.85$. 

In [ ]:
# NOTE:
#  - this NB focuses on the fit of the detector spatial resolution kernel parameters
#  - the fit is performed by taking the kernel, which follows a modsech template, and leave only a single free parameter
#  - here the fit procedure is automated to account for MULTIPLE sources and MULTIPLE energy bands

from singleCAM_IROS._pipeline_support import _handle_dirpaths
from typing import Callable

from pathlib import Path

from numpy.typing import NDArray
import numpy as np
from astropy.io.fits.fitsrec import FITS_rec

from bloodmoon.coords import equatorial2shift
from bloodmoon.io import simulation_files
from bloodmoon.mask import CodedMaskCamera, codedmask, count
from bloodmoon.images import argmax

import darksun as ds

ds.show.set_figures_darkbkg()

In [11]:
MASK_FITS: str = "wfm_mask_NTHT_20250725.fits"

SKYFIELD: str = "IROSDummy"
DATA_FITS: str = "iros_benchmark_2-50keV_mask_050_1040x17_infdet_1ks"

ID_CAMERA_A: str = "cam1a"
ID_CAMERA_B: str = "cam1b"

UPS_X: int = 10
UPS_Y: int = 5

VIGNETTING: bool = True

In [12]:
# load filepaths
mask_path, simul_data, save_path = _handle_dirpaths(
    mask=MASK_FITS,
    skyfield=SKYFIELD,
    simul=DATA_FITS,
)
wfm: CodedMaskCamera = codedmask(mask_path, UPS_X, UPS_Y)
filepaths: dict[str, dict[str, Path]] = simulation_files(simul_data)


# data from camera A
catalogueA = ds.get_catalogue(filepaths[ID_CAMERA_A]['sources'])
#sdlA_detct = ds.get_data(filepaths[ID_CAMERA_A]['detected'])
sdlA_recnstr = ds.get_data(filepaths[ID_CAMERA_A]['reconstructed'])

## data from camera B
catalogueB = ds.get_catalogue(filepaths[ID_CAMERA_B]['sources'])
#sdlB_detct = ds.get_data(filepaths[ID_CAMERA_B]['detected'])
sdlB_recnstr = ds.get_data(filepaths[ID_CAMERA_B]['reconstructed'])

In [13]:
from bloodmoon.types import CoordEquatorial
from darksun.benchmarking import source_catalogue_data
from darksun.data import DataLoader, CatalogueLoader

type EnergyRange = tuple[float, float]


def get_source_coords(
    sourceID: str,
    catalogue: CatalogueLoader,
) -> CoordEquatorial:
    """Extracts the source RA/Dec coords from catalogue."""
    data = source_catalogue_data(sourceID, catalogue.DLdata)
    return CoordEquatorial(data['RA'], data['DEC'])

def filter_energy(
    photons: FITS_rec,
    eband: EnergyRange,
) -> FITS_rec:
    """Filters the input photons by energy, in [keV]."""
    E_min, E_max = eband
    return ds.filter_data(photons, E_min=E_min, E_max=E_max, coords=None)

In [14]:
from scipy.signal import convolve
from bloodmoon.images import fshift
from bloodmoon.optim import _detector_footprint_cached, apply_vignetting


def project_mask_pattern(
    camera: CodedMaskCamera,
    shift_x: float,
    shift_y: float,
    vignetting: bool,
) -> NDArray:
    """Projects the camera mask pattern on the detector plane."""
    pxdimy, pxdimx = (
        camera.specs.mask_deltay / camera.upscale_f.y,
        camera.specs.mask_deltax / camera.upscale_f.x,
    )
    fr, fc = (
        (-1.0) * shift_y / pxdimy,
        (-1.0) * shift_x / pxdimx,
    )
    mask_shifted = fshift(camera.mask.astype(float), fr, fc)
    mask_vignetted = (
        apply_vignetting(camera, mask_shifted, shift_x, shift_y)
        if vignetting else mask_shifted
    )
    return mask_vignetted

def extract_detector(
    camera: CodedMaskCamera,
    shadowgram: NDArray,
) -> NDArray:
    """
    Extracts the detector image from the mask pattern projection on the detector plane.
    """
    i_min, i_max, j_min, j_max = _detector_footprint_cached(camera)
    detector = shadowgram[i_min:i_max, j_min:j_max]
    detector *= camera.bulk
    detector /= np.sum(detector)
    return detector


def compute_psfy_kernel(x: NDArray, alpha: float, beta: float) -> NDArray:
    """Computes the LEM-X detector spatial resolution kernel from input params."""
    modsec = 1.0 / np.cosh(np.abs(x / alpha) ** beta)
    psfy = modsec.reshape(len(x), -1)
    return psfy / np.sum(psfy)

def apply_detector_resolution(shadowgram: NDArray, kernel: NDArray) -> NDArray:
    """Applies finite detector spatial resolution effects to a shadowgram."""
    return convolve(shadowgram, kernel, mode="same")

In [ ]:
from typing import NamedTuple
from functools import partial
from scipy.optimize import curve_fit

class OptResult(NamedTuple):
    """Optimisation result."""
    params: NDArray
    errs: NDArray


def config_KernelFitFunc(
    camera: CodedMaskCamera,
    shift_x: float,
    shift_y: float,
    cts: float,
    vignetting: bool,
) -> Callable[[NDArray, float], NDArray]:
    """Initialises the kernel model."""
    # we take a whole slit to have a good kernel spatial extension
    px_ydim = camera.specs.mask_deltay / camera.upscale_f.y
    slit_dim = camera.specs.slit_deltay
    # the kernel must have the same binning as the mask elements
    bins = np.linspace(-slit_dim, slit_dim, int(2 * slit_dim / px_ydim) + 1)
    make_kernel: Callable = partial(compute_psfy_kernel, bins)
    # compute projected source shadowgram (non normalised)
    sg = project_mask_pattern(camera, shift_x, shift_y, vignetting)

    def f(x: NDArray, alpha: float, beta: float = 0.85) -> NDArray:
        """Models the PSFY kernel and performs the convolution with detected array."""
        kernel = make_kernel(alpha, beta)
        conv = apply_detector_resolution(sg, kernel)
        detector = extract_detector(camera, conv) * cts
        return detector.flatten()
    
    return f


def kernel_params_fit(
    camera: CodedMaskCamera,
    shift_x: float,
    shift_y: float,
    cts: float,
    reconstructed: NDArray,
    vignetting: bool = True,
    verbose: bool = True,
) -> OptResult:
    """Performs the fit of the detector spatial resolution kernel params."""
    func = config_KernelFitFunc(camera, shift_x, shift_y, cts, vignetting)

    # - setup ydata and p0 from current kernel (alpha, beta)
    ydata = reconstructed.flatten()
    start_params_vals = np.array([0.54597])
    # - setup boundaries and least squares kwargs
    bounds = [
        (0.0,), (1.0,),
    ]
    lsq_kwargs = {
        'jac': '3-point',
        'ftol': 1e-8,
        'xtol': 1e-8,
        'x_scale': 'jac',
        'loss': 'linear',
    }
    
    popt, pcov, info, msg, iflag = curve_fit(
        f=func,
        xdata=np.arange(len(ydata)),
        ydata=ydata,
        p0=start_params_vals,
        bounds=bounds,
        method='trf',
        full_output=True,
        **lsq_kwargs,
    )
    errs = np.sqrt(np.diag(pcov))

    if verbose:
        routine_status = [
            'Convergence in chi-square values',
            'Convergence in parameter values',
            'Convergence in both chi-square and parameter values',
            'Convergence in orthogonality',
        ]
        print('## Optimisation Results:')
        for idx, (p, dp) in enumerate(zip(popt, errs)):
            print(f'  - p[{idx}] OPTIM.: {float(p):.7f} +/- {float(dp):.7f}')
        print(
            f'\n## Fit Report:\n'
            f'  - func calls (also # of iters): {info['nfev']}\n'
            f'  - procedure msg (`ftol`, `xtol` = {lsq_kwargs['ftol']}, {lsq_kwargs['xtol']}): {msg}\n'
            f'  - success status {iflag}: {routine_status[iflag - 1]}\n'
        )

    return OptResult(popt, errs)

In [ ]:
def fit_kernel(
    camera: CodedMaskCamera,
    sourcesID: tuple[str, ...],
    energy_bands: tuple[EnergyRange, ...],
    sdl: DataLoader,
    catalogue: CatalogueLoader,
    vignetting: bool = True,
    verbose: bool = True,
) -> dict[str, dict[EnergyRange, OptResult]]:
    """
    Performs the PSFY kernel fit for photons coming from given directions and in given energy bands.
    """
    def fit_across_ebands(
        coords: CoordEquatorial,
        src_phs: FITS_rec,
    ) -> dict[EnergyRange, OptResult]:
        """
        Performs the fit of the PSFY kernel for a source in given energy bands.
        """
        out: dict = {}
        shift_x, shift_y = equatorial2shift(sdl, camera, coords.ra, coords.dec)
        for eband in energy_bands:
            print(f'#### Analysing energy band {eband}...')
            eband_phs = filter_energy(src_phs, eband)
            detector = count(camera, eband_phs)[0]
            cts = 0.8 * detector.sum()                     # NOTE!!!  THIS IS A PROBLEM
            params: OptResult = kernel_params_fit(
                camera, shift_x, shift_y, cts, detector, vignetting, verbose,
            )
            out[eband] = params

        return out
    
    out: dict = {}
    for src in sourcesID:
        print(f'\n#### ---- Analysing source {src.upper()}...\n')
        coords: CoordEquatorial = get_source_coords(src, catalogue)
        src_phs_list: FITS_rec = ds.select_source_photons(coords, sdl.DLdata, False)
        results = fit_across_ebands(coords, src_phs_list)
        out[src] = results

    return out

In [17]:
sourcesID: tuple[str, ...] = (
    's17', 's25',
)
ebands: tuple[EnergyRange, ...] = (
    (2.0, 3.0), (4.0, 5.0), (7.0, 8.0),
)
results: dict[str, dict[EnergyRange, OptResult]] = fit_kernel(
    wfm, sourcesID, ebands, sdlA_recnstr, catalogueA, VIGNETTING,
)


#### ---- Analysing source S17...
#### Analysing energy band (2.0, 3.0)...
## USING BULK MASK with 1.5 mm cover ##
## Optimisation Results:
  - p[0] OPTIM.: 95075.1223725 +/- 146.9901218
  - p[1] OPTIM.: 1.0000000 +/- 0.0089873

## Fit Report:
  - func calls (also # of iters): 7
  - procedure msg (`ftol`, `xtol` = 1e-08, 1e-08): `ftol` termination condition is satisfied.
  - success status 2: Convergence in parameter values

#### Analysing energy band (4.0, 5.0)...
## Optimisation Results:
  - p[0] OPTIM.: 103549.9983976 +/- 146.7493203
  - p[1] OPTIM.: 0.7775237 +/- 0.0070809

## Fit Report:
  - func calls (also # of iters): 6
  - procedure msg (`ftol`, `xtol` = 1e-08, 1e-08): `ftol` termination condition is satisfied.
  - success status 2: Convergence in parameter values

#### Analysing energy band (7.0, 8.0)...
## Optimisation Results:
  - p[0] OPTIM.: 45387.3346715 +/- 93.4197983
  - p[1] OPTIM.: 0.5002394 +/- 0.0079094

## Fit Report:
  - func calls (also # of iters): 6
  - proce

In [18]:
results

{'s17': {(2.0,
   3.0): OptResult(params=array([9.50751224e+04, 1.00000000e+00]), errs=array([1.46990122e+02, 8.98732634e-03])),
  (4.0,
   5.0): OptResult(params=array([1.03549998e+05, 7.77523686e-01]), errs=array([1.46749320e+02, 7.08093999e-03])),
  (7.0,
   8.0): OptResult(params=array([4.53873347e+04, 5.00239408e-01]), errs=array([9.34197983e+01, 7.90942954e-03]))},
 's25': {(2.0,
   3.0): OptResult(params=array([2.94192625e+04, 1.00000000e+00]), errs=array([1.19338309e+02, 2.35569269e-02])),
  (4.0,
   5.0): OptResult(params=array([2.97623773e+04, 1.00000000e+00]), errs=array([1.18811373e+02, 2.31825354e-02])),
  (7.0,
   8.0): OptResult(params=array([1.32127879e+04, 9.46110225e-01]), errs=array([7.83362057e+01, 3.32413493e-02]))}}

In [8]:
# NOTE: just a check on cts values

sourceID: str = 's25'

coords: CoordEquatorial = get_source_coords(sourceID, catalogueA)
phs: FITS_rec = ds.select_source_photons(coords, sdlA_recnstr.DLdata, False)

In [9]:
from bloodmoon.mask import decode

eband: EnergyRange = (2.0, 3.0)  # [keV]

eband_phs: FITS_rec = filter_energy(phs, eband)
shift_x, shift_y = equatorial2shift(sdlA_recnstr, wfm, coords.ra, coords.dec)

reconstructed: NDArray = count(wfm, eband_phs)[0]
sky_reconstructed = decode(wfm, reconstructed)

## USING BULK MASK with 1.5 mm cover ##


In [16]:
0.8 * reconstructed.sum() / sky_reconstructed.max()

np.float64(1.360697178338697)